In [ ]:
from PIL import Image as PIL
from pdf417decoder import PDF417Decoder

image = PIL.open(r"C:\Users\FABLAB EAN\Pictures\test.png")
decoder = PDF417Decoder(image)

if (decoder.decode() > 0):
    decoded = decoder.barcode_data_index_to_string(0)

: 

In [ ]:
import cv2
from PIL import Image
from pdf417decoder import PDF417Decoder
import numpy as np

cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)

x1,y1= 150, 150
x2,y2= 800, 270


while True:

    ret, frame = cap.read()

    cv2.rectangle(frame,(x1,y1),(x2,y2),(0, 255, 0), 2)
    cv2.imshow("Captura", frame)


    key = cv2.waitKey(1)

    # Presiona S para intentar leer el código
    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        # Convertir el frame de OpenCV (BGR) a PIL (RGB)
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(img_rgb)

        decoder = PDF417Decoder(pil_image)
        result = decoder.decode()
        print("Códigos detectados:", result)

        if result > 0:
            decoded = decoder.barcode_data_index_to_string(0)
            print("\n===== DATOS DECODIFICADOS =====")
            print(decoded)
        else:
            print("No se encontró ningún PDF417 en la captura.")

    # Presiona ESC para salir
        if key == 27:
            break


cap.release()
cv2.destroyAllWindows()


Códigos detectados: 0
No se encontró ningún PDF417 en la captura.
Códigos detectados: 0
No se encontró ningún PDF417 en la captura.
Códigos detectados: 0
No se encontró ningún PDF417 en la captura.
Códigos detectados: 0
No se encontró ningún PDF417 en la captura.
Códigos detectados: 0
No se encontró ningún PDF417 en la captura.
Códigos detectados: 0
No se encontró ningún PDF417 en la captura.


KeyboardInterrupt: 

: 

In [2]:


from PIL import Image
from pdf417decoder import PDF417Decoder

img = Image.open(r"C:\Users\FABLAB EAN\Pictures\test.png")
decoder = PDF417Decoder(img)
result = decoder.decode()
print("Códigos detectados:", result)

if result > 0:
    print(decoder.barcode_data_index_to_string(0))

Códigos detectados: 0


In [6]:
import cv2
import zxingcpp

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))


x1, y1 = 500, 150
x2, y2 = 1700, 800


while True:
    ret, frame = cap.read()

    # Dibujar recuadro verde
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    # Presiona S para leer
    if key == ord("s"):

        # Recortar SOLO el área definida
        cropped = frame[y1:y2, x1:x2]

        # Convertir a RGB (ZXing recibe en RGB)
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        # Leer códigos con ZXing
        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")
            for r in results:
                print(f"Formato: {r.format}")
                print(f"Datos  : {r.text}\n")

    # ESC para salir
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()


Resolución real: 1920.0 x 1080.0

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.DataBar
Datos  : 36850578271822

No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.PDF417
Datos  : 0359162481<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>PubDSK_1<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>490037<NUL><NUL>1070023664RAMIREZ<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>ROBAYO<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>JULIAN<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>0M19991202153400O+<NUL>2<STX>C<NUL><U+8B>Tÿ<U+80><U+80>r«{<U+82>f<U+81>U<U+90>[<U+95>ÖXÙkÊJÃ[Vg¬]¸U<U+9E>e¦`·K½=ÁIfg®>fO<U+85>)<U+97>^£U§@}c<U+84>4<U+9C>E<U

In [ ]:
import cv2
import zxingcpp
import re

# ===============================
# Función para limpiar y extraer datos de la cédula
# ===============================
def parse_pdf417(text):
    # Quitar caracteres no imprimibles
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)


    data = {}

    # ===============================
    # 1. Encontrar TODAS las cadenas numéricas de 10 dígitos
    # ===============================
    all_10_digits = re.findall(r'(?<!\d)\d{10}(?!\d)', clean)

    # Regla:
    # - Ignorar la primera
    # - Si existe segunda → esa es la cédula
    if len(all_10_digits) >= 2:
        data["cedula"] = all_10_digits[1]  # segunda coincidencia
    else:
        data["cedula"] = None

    # ===============================
    # 2. Ignorar cadenas de 6 dígitos (NO HACER NADA CON ELLAS)
    # (no se guardan, solo se ignoran)
    # ===============================
    # Patrón: \b\d{6}\b
    # Solo las omitimos y seguimos

    # ===============================
    # 3. Buscar sexo + fecha (MYYYYMMDD / FYYYYMMDD)
    # ===============================
    m = re.search(r'([MF])(\d{8})', clean)
    if m:
        data["sexo"] = m.group(1)
        data["fecha_nac"] = m.group(2)

    # ===============================
    # 4. Buscar RH
    # ===============================
    m = re.search(r'(A|B|O)[+-]', clean)
    if m:
        data["rh"] = m.group(0)

    # ===============================
    # 5. Extraer apellidos y nombre
    # ===============================
    m = re.search(r'(\d{10})([A-ZÑÁÉÍÓÚ]+)([A-ZÑÁÉÍÓÚ]+)([A-ZÑÁÉÍÓÚ]+).*?[MF]\d{8}', clean)
    if m:
        data["apellido1"] = m.group(2)
        data["apellido2"] = m.group(3)
        data["nombre"]   = m.group(4)

    return clean, data


# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


Resolución real: 1920.0 x 1080.0
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.DataBar

--- Texto limpio ---
04563935523403

--- Datos extraídos ---
cedula: None


No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.PDF417

--- Texto limpio ---
0359162481<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>PubDSK_1<NUL><NUL><NUL><NUL><NUL><NU

In [1]:
import cv2
import zxingcpp
import json

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):
        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código en el recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")
                print(f"Datos  : {r.text}\n")

                # ---------------------------------------------------
                # Detectar PDF417
                # ---------------------------------------------------
                if r.format == zxingcpp.BarcodeFormat.PDF417:
                    print("→ Se detectó PDF417 de la cédula")

                # ---------------------------------------------------
                # Detectar QR
                # ---------------------------------------------------
                if r.format == zxingcpp.BarcodeFormat.QR_CODE:
                    print("→ Se detectó QR de la nueva cédula")

                    try:
                        qr_json = json.loads(r.text)
                        print("===== DATOS DEL QR =====")
                        for k, v in qr_json.items():
                            print(f"{k}: {v}")
                    except:
                        print("El QR no contiene JSON o está protegido/codificado")

    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()


Resolución real: 1920.0 x 1080.0
No se encontró ningún código en el recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.QRCode
Datos  : <DC4>õ<U+97><DEL>Ò<U+8A><US>_¤<U+90>º<ETB>w[ñ<U+86><SYN>ØR$¼<U+80>§`<ESC><U+86><U+8D><U+9E><FS>,Uö*<ACK>Nÿw<CAN>h_!¡þocu¥<U+9B>><NUL><HT>^<FS>lã<U+9A><SO>¨!<U+99>±J½BºWÓÞz=ïQ~X¿<ACK>j<SUB><GS>ñ<DC4><SOH>[<FF>C½S<U+88>Ûÿ<FS><U+96>ìùÉôÎõ<U+96>*E-5Â2bBÊ<SO>¿æ­<DC1>F<FF>¡³<U+8D>Íç/<DC1><FS>*Bg<U+86><BS>goM<DC1>-<CR>GòÕ<U+8B>ÎóÿÒ<SOH><SO>äN­PGn<ETB><ACK>¹D#c.7~<U+8B>S-<BEL><U+89><CAN><U+81>dZ}<FS>¹<DC1>Bô~<BEL>1<U+89><U+8A>{æÇ<U+93>¼<U+91>¶sHwe<U+90>!ò@YÐ©ä<U+90>ñ+k¿¬Á»<U+84>È<DLE>}'?ò<U+9B>Pû#f<U+98>L/<FS>ÃÎ3¢<CR>¯.<U+8E>åC<U+9B>a]þ<VT>ññ;¿#¶<BEL>{Ñ.iíä<ETX>éæ]<NUL>In<U+82>Z<US>¿[ÊÎ<ETX>èæÜ<NAK>¢¾ÀéLê<U+92>×p`I<U+89>|<GS><ENQ>^fÉ<U+90>·PS+m//<BS><U+8D><U+86>¦Ôm<U+9F>~<U+85>®÷â<FS>d<EM>U<U+8A>i<SUB>ÕXû2£2ò<SI>øenñr<U+9F>J<ACK><è'<SYN><RS><VT><U+99>mF<U+94><ETX>ÉUéÍ¡Ò<RS>øS<U+86><U+84>ÝgUDÁÏ"<DC1><U+99>ÿæªy=àü§Àã<RS>dÔ­eÈ<STX>`êþ¾Ë@<DEL>jE<NUL>\

AttributeError: type object 'zxingcpp.BarcodeFormat' has no attribute 'QR_CODE'

In [5]:
import cv2
import zxingcpp
import re

# ===============================
# Función para limpiar y extraer datos de la cédula
# ===============================
def parse_pdf417(text):
    # Quitar caracteres no imprimibles
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)
    clean = re.sub(r'\s+', ' ', clean)

    data = {}

    # ===============================
    # 1. Encontrar TODAS las cadenas numéricas de 10 dígitos
    # ===============================
    all_10_digits = re.findall(r'(?<!\d)\d{10}(?!\d)', clean)

    # Regla:
    # - Ignorar la primera
    # - Si existe segunda → esa es la cédula
    if len(all_10_digits) >= 2:
        data["cedula"] = all_10_digits[1]  # segunda coincidencia
    else:
        data["cedula"] = None

    # ===============================
    # 2. Ignorar cadenas de 6 dígitos (NO HACER NADA CON ELLAS)
    # (no se guardan, solo se ignoran)
    # ===============================
    # Patrón: \b\d{6}\b
    # Solo las omitimos y seguimos

    # ===============================
    # 3. Buscar sexo + fecha (MYYYYMMDD / FYYYYMMDD)
    # ===============================
    m = re.search(r'([MF])(\d{8})', clean)
    if m:
        data["sexo"] = m.group(1)
        data["fecha_nac"] = m.group(2)

    # ===============================
    # 4. Buscar RH
    # ===============================
    m = re.search(r'(A|B|O)[+-]', clean)
    if m:
        data["rh"] = m.group(0)

    # ===============================
    # 5. Extraer apellidos y nombre
    # ===============================
    m = re.search(r'\d{10}\s+([A-ZÑÁÉÍÓÚ]+)\s+([A-ZÑÁÉÍÓÚ]+)\s+([A-ZÑÁÉÍÓÚ]+)', clean)
    if m:
        data["apellido1"] = m.group(2)
        data["apellido2"] = m.group(3)
        data["nombre"]   = m.group(4)

    return clean, data


# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


Resolución real: 1920.0 x 1080.0
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.DataBar

--- Texto limpio ---
02022300222058

--- Datos extraídos ---
cedula: None


No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.PDF417

--- Texto limpio ---
0359162481<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>PubDSK_1<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>490037<NUL><NUL>1070023664RAMIREZ<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>ROBAYO<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>JULIAN<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NU

In [7]:
import cv2
import zxingcpp
import re
import unicodedata



def limpiar(texto):
    # Quitar caracteres no ASCII imprimibles
    texto = ''.join(c for c in texto if 32 <= ord(c) <= 126)
    return texto

def extraer_datos_pdf417(raw):
    txt = limpiar(raw)

    # Buscar la cédula (10 dígitos)
    m = re.search(r'(\d{10})', txt)
    if not m:
        return None
    
    inicio = m.start()
    bloque = txt[inicio:]    # Tomamos desde la cédula hacia adelante

    # Longitudes en el PDF417 real
    long_cedula = 10
    long_ap1    = 15
    long_ap2    = 15
    long_nom    = 15

    cedula = bloque[0:10]

    ap1   = bloque[10:10+15].strip()
    ap2   = bloque[25:25+15].strip()
    nom   = bloque[40:40+15].strip()

    # Sexo, fecha, RH
    sexo = bloque[55]           # M / F
    fecha = bloque[56:64]       # YYYYMMDD
    rh = bloque[64:67].replace("<","").strip()

    return {
        "cedula": cedula,
        "apellido1": ap1,
        "apellido2": ap2,
        "nombre": nom,
        "sexo": sexo,
        "fecha_nac": fecha,
        "rh": rh
    }

# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


Resolución real: 1920.0 x 1080.0
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.PDF417

--- Texto limpio ---
0359162481<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>PubDSK_1<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>490037<NUL><NUL>1070023664RAMIREZ<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>ROBAYO<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>JULIAN<NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL><NUL>0M19991202153400O+<NUL>2<STX>C<NUL><U+8B>Tÿ<U+80><U+80>r«{<U+82>f<U+81>U<U+90>[<U+95>ÖXÙkÊJÃ[Vg¬]¸U<U+9E>e¦`·K½=ÁIfg®>fO<U+85>)<U+97>^£U§@}c<U+84>4<U+9C>E<U+A0>-<U+82>[s¶}£<U+89><U+91><U+8A>¯<U+8E><U+98>c<U+90><U+8E><U+8E>J¡r<U+85>

In [ ]:
import cv2
import zxingcpp
import re

# ===============================
# Función para limpiar y extraer datos de la cédula
# ===============================
def parse_pdf417(text):
    # Quitar caracteres no imprimibles
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)

    # Quitar palabras "NUL" que vienen del decodificador
    clean = clean.replace("NUL", " ")

    data = {}

    # ===============================
    # 1. Encontrar las cadenas numéricas de 10 dígitos
    # ===============================
    all_10_digits = re.findall(r'(?<!\d)\d{10}(?!\d)', clean)

    if len(all_10_digits) >= 2:
        cedula = all_10_digits[1]
        data["cedula"] = cedula
    else:
        data["cedula"] = None
        cedula = None

    # ===============================
    # 2. Sexo + fecha
    # ===============================
    m = re.search(r'([MF])(\d{8})', clean)
    if m:
        data["sexo"] = m.group(1)
        data["fecha_nac"] = m.group(2)

    # ===============================
    # 3. RH
    # ===============================
    m = re.search(r'(A|B|O)[+-]', clean)
    if m:
        data["rh"] = m.group(0)

    # ===============================
    # 4. Apellidos y nombre SIN “NUL”
    # ===============================
    if cedula:
        pos = clean.find(cedula)
        tail = clean[pos + len(cedula):]

        # Grupos reales de letras (mínimo 2 letras)
        grupos = re.findall(r'\b[A-ZÑÁÉÍÓÚ]{2,}\b', tail)

        # Evitar basura
        grupos = [g for g in grupos if g not in ["N", "NU", "NUL"]]

        if len(grupos) >= 1:
            data["apellido1"] = grupos[0]
        if len(grupos) >= 2:
            data["apellido2"] = grupos[1]
        if len(grupos) >= 3:
            data["nombre"] = grupos[2]

    return clean, data



# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


Resolución real: 1920.0 x 1080.0
No se encontró ningún código PDF417 dentro del recuadro.

===== CÓDIGOS DETECTADOS =====
Formato: BarcodeFormat.PDF417

--- Texto limpio ---
0359162481< >< >< >< >< >< >< >< >< >< >< >< >< >< >PubDSK_1< >< >< >< >< >< >< >< >490037< >< >1070023664RAMIREZ< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >ROBAYO< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >JULIAN< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >< >0M19991202153400O+< >2<STX>C< ><U+8B>Tÿ<U+80><U+80>r«{<U+82>f<U+81>U<U+90>[<U+95>ÖXÙkÊJÃ[Vg¬]¸U<U+9E>e¦`·K½=ÁIfg®>fO<U+85>)<U+97>^£U§@}c<U+84>4<U+9C>E<U+A0>-<U+82>[s¶}£<U+89><U+91><U+8A>¯<U+8E><U+98>c<U+90><U+8E><U+8E>J¡r<U+85><U+95><U+81><U+97>´<U+9B>®<U+9D><U+8F><U+98>À¡<U+A0>E<U+8E>a<U+80><U+97>y<U+A0>¯¥<U+86>©<U+9E>oz°<U+8E>³©¡y¹µ§Å¶<U+83>»£¾<U+8C>»sÚ<U+8F>ScXf¹N<U+89>8fW~0oå<U+9F><NAK>A<ENQ>aÿ<U+85><U+84>T`$ÉËòÿ< >7<STX>C< ><U+86>Rÿ<U+80><U+80>ys^®;<U+97>P<U+81>={¼kÒiÈaYlO